# Hospital Policy QA: a LoRA fine-tune of Qwen2.5-0.5B-Instruct

A small assistant that answers staff questions about hospital policy, using only the policy passage it is given. For example: *"What Braden Scale score means a patient is at risk?"* → *"18 or lower"*.

**What this notebook does**
1. Builds 34 question-answer pairs from public guidance on six topics (CDC, WHO, AHRQ).
2. Holds out one whole topic (pressure injuries) so the test uses a policy the model never saw.
3. Asks the **base model** the held-out questions, then fine-tunes it with LoRA, then asks again.
4. Shows the answers side by side, scored with exact match and token F1.

Run the cells top to bottom on a GPU runtime (`Runtime > Change runtime type > T4 GPU`). The whole run takes a few minutes on a free T4.

In [ ]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate

## Data

34 question-answer pairs written from public guidance: CDC STEADI fall prevention, WHO hand hygiene, CDC sepsis recognition, SBAR handoff communication (AHRQ / TeamSTEPPS), CDC CLABSI prevention and AHRQ pressure-injury prevention.

**Honest test split:** all 5 pressure-injury questions are held out. The model never sees that passage or those questions during training, so the final test measures whether it learned the *skill* (answering briefly from a passage), not the answers themselves.

In [ ]:
FALL_CTX = "Our hospital's fall-prevention protocol follows the CDC's STEADI framework, which has three steps: Screen, Assess, and Intervene. Staff screen every patient age 65 and older by asking whether they have fallen in the past year, feel unsteady while standing or walking, or worry about falling. Patients who screen positive are assessed using standardized tests: the Timed Up and Go test, the 30-second chair stand test, the 4-stage balance test, and a check of orthostatic blood pressure. Staff also review the patient's medications for drugs linked to increased fall risk. Based on the assessment, each at-risk patient gets a personalized fall-prevention plan, following the AHRQ Fall TIPS model of assessing risk, building a plan, and executing it consistently. Interventions include posting a fall-risk alert at the bedside, scheduled rounding to offer toileting assistance, keeping the bed in the lowest position with brakes locked, and ensuring the call light is within reach. More than one in four adults 65 and older falls each year, and hospitalized older adults are especially likely to fall in the first month after discharge, so the plan is also communicated to outpatient follow-up providers at discharge."
HYGIENE_CTX = "Hand hygiene in our hospital follows the World Health Organization's My Five Moments for Hand Hygiene framework, which was developed to interrupt the transmission of germs between healthcare workers, patients, and the care environment. Staff must clean their hands at five specific moments: before touching a patient, before a clean or aseptic procedure, after a procedure or exposure to body fluids, after touching a patient, and after touching a patient's immediate surroundings even if the patient itself was not touched. The first moment protects the patient from germs on the worker's hands, while the other four moments mainly protect the healthcare worker and the surrounding environment from contamination. This framework combines the hand hygiene indications from both the WHO Guidelines on Hand Hygiene in Health Care and the CDC's Guideline for Hand Hygiene in Health Care Settings into a single, easy-to-remember set of moments staff can apply during any patient care sequence."
SEPSIS_CTX = "Our sepsis protocol is based on CDC's Get Ahead of Sepsis guidance: know the risk, spot the signs, and act fast. Sepsis is the body's extreme, life-threatening response to an infection, so any suspected case is treated as a medical emergency requiring immediate evaluation. Staff watch for signs including confusion or disorientation, shortness of breath, clammy or sweaty skin, extreme pain or discomfort, fever with shivering or feeling very cold, and a high heart rate or weak pulse. Clinicians confirm suspected sepsis using physical findings such as fever, increased heart rate, low blood pressure, and trouble breathing, together with tests for infection or organ damage. When any staff member suspects sepsis, they must immediately notify the care team and follow the hospital's sepsis protocol rather than waiting for confirmation, since early recognition and prompt treatment significantly improve patient outcomes."
SBAR_CTX = "Our hospital uses the SBAR framework for handoff communication: Situation, Background, Assessment, and Recommendation. SBAR was first developed at Kaiser Permanente and is now endorsed by AHRQ, IHI, and WHO as a standard handoff tool. Situation states what is currently happening with the patient, usually starting with the identity of the person communicating, patient identifiers such as age and gender, and a brief statement of the current problem. Background covers the patient's clinical history related to the current situation, presenting signs and symptoms, and relevant test results such as labs or imaging. Assessment states what the person communicating thinks the problem is, based on the background information, patient history, and observations, and can include objective data such as vital signs. The final step has the speaker state an initial recommendation or request, including what is needed and when, and then repeat back the other provider's response to confirm accuracy. Staff are also reminded to introduce themselves at the start of every SBAR call, since not everyone on the line may know who they are."
CLABSI_CTX = "Our central line insertion and maintenance protocol follows the CDC's central-line-associated bloodstream infection, or CLABSI, prevention bundle. Before insertion, staff perform hand hygiene and then use maximal sterile barrier precautions, including a cap, mask, sterile gown, sterile gloves, and a full-body sterile drape. The skin is prepped with a chlorhexidine-based antiseptic, since trials show chlorhexidine-alcohol outperforms povidone-iodine-alcohol at preventing CLABSI. Staff avoid the femoral site when possible when choosing where to place the catheter. Hand hygiene is also required before and after touching the insertion site, and whenever the line is accessed, repaired, or redressed. Every line is reviewed daily, and any catheter that is no longer necessary is removed promptly, since bundling these measures together has produced large drops in CLABSI rates."
PRESSURE_CTX = "Our hospital's pressure-injury prevention protocol follows AHRQ's toolkit approach: comprehensive skin assessment, standardized risk assessment, and care planning to address identified risks. Nursing staff use the Braden Scale to screen every patient, and a score of 18 or lower indicates the patient is at risk. Care planning follows the assessment, and any subscale that scores abnormally must be addressed in the care plan even if the overall score is not low. Staff reposition at-risk patients frequently, with many protocols calling for repositioning every two hours and reassessment with the Braden Scale every shift. Prevention is a team effort that also involves physical and occupational therapists and dietary staff, since nutrition affects skin integrity."

qa_pairs = [(FALL_CTX, "What framework does our fall-prevention protocol follow?", "the CDC's STEADI framework"), (FALL_CTX, "What are the three steps of the STEADI framework?", "Screen, Assess, and Intervene"), (FALL_CTX, "What age do staff begin screening patients for fall risk?", "65"), (FALL_CTX, "What test checks a patient's blood pressure while standing versus lying down?", "orthostatic blood pressure"), (FALL_CTX, "What model guides building a personalized fall-prevention plan?", "the AHRQ Fall TIPS model"), (FALL_CTX, "Where is a fall-risk alert posted?", "at the bedside"), (FALL_CTX, "What fraction of adults 65 and older falls each year?", "more than one in four"), (FALL_CTX, "When are hospitalized older adults especially likely to fall?", "in the first month after discharge"), (HYGIENE_CTX, "What framework does our hospital's hand hygiene policy follow?", "the World Health Organization's My Five Moments for Hand Hygiene framework"), (HYGIENE_CTX, "How many specific moments require hand hygiene?", "five"), (HYGIENE_CTX, "Which moment protects the patient specifically?", "before touching a patient"), (HYGIENE_CTX, "What should staff do after touching a patient's surroundings, even without touching the patient?", "clean their hands"), (HYGIENE_CTX, "Which two guidelines does the five-moments framework combine?", "the WHO Guidelines on Hand Hygiene in Health Care and the CDC's Guideline for Hand Hygiene in Health Care Settings"), (SEPSIS_CTX, "What is the CDC guidance our sepsis protocol is based on?", "Get Ahead of Sepsis"), (SEPSIS_CTX, "What are sepsis's three core guidance steps?", "know the risk, spot the signs, and act fast"), (SEPSIS_CTX, "What is sepsis?", "the body's extreme, life-threatening response to an infection"), (SEPSIS_CTX, "What should staff do when they suspect sepsis?", "immediately notify the care team and follow the hospital's sepsis protocol"), (SEPSIS_CTX, "What vital sign findings do clinicians check for sepsis?", "fever, increased heart rate, low blood pressure, and trouble breathing"), (SBAR_CTX, "What does SBAR stand for?", "Situation, Background, Assessment, and Recommendation"), (SBAR_CTX, "Where was SBAR first developed?", "Kaiser Permanente"), (SBAR_CTX, "Which three organizations endorse SBAR as a standard handoff tool?", "AHRQ, IHI, and WHO"), (SBAR_CTX, "What does the Background step cover?", "the patient's clinical history related to the current situation, presenting signs and symptoms, and relevant test results such as labs or imaging"), (SBAR_CTX, "What should staff do at the start of every SBAR call?", "introduce themselves"), (CLABSI_CTX, "What does CLABSI stand for?", "central-line-associated bloodstream infection"), (CLABSI_CTX, "What must staff do immediately before inserting a central line?", "perform hand hygiene"), (CLABSI_CTX, "What does maximal sterile barrier precaution include?", "a cap, mask, sterile gown, sterile gloves, and a full-body sterile drape"), (CLABSI_CTX, "What antiseptic is used to prep the skin before catheter insertion?", "a chlorhexidine-based antiseptic"), (CLABSI_CTX, "Which insertion site should be avoided when possible?", "the femoral site"), (CLABSI_CTX, "How often is every line reviewed for continued necessity?", "daily"), (PRESSURE_CTX, "What scale do nursing staff use to screen for pressure-injury risk?", "the Braden Scale"), (PRESSURE_CTX, "What Braden Scale score indicates a patient is at risk?", "18 or lower"), (PRESSURE_CTX, "How often should at-risk patients typically be repositioned?", "every two hours"), (PRESSURE_CTX, "Besides nurses, which staff contribute to pressure-injury prevention?", "physical and occupational therapists and dietary staff"), (PRESSURE_CTX, "What three things does AHRQ's toolkit checklist call for?", "comprehensive skin assessment, standardized risk assessment, and care planning to address identified risks")]

HELD_OUT_TOPIC = PRESSURE_CTX
train_pairs = [p for p in qa_pairs if p[0] != HELD_OUT_TOPIC]
test_pairs = [p for p in qa_pairs if p[0] == HELD_OUT_TOPIC]

def make_prompt(context, question):
    return (
        "Below is a passage and a question about it. Answer the question using only the passage.\n\n"
        f"### Passage:\n{context}\n\n"
        f"### Question:\n{question}\n\n"
        "### Answer:\n"
    )

print(f"{len(train_pairs)} training pairs, {len(test_pairs)} held-out test pairs")

## Load the base model in 4-bit

4-bit quantization keeps the 0.5B model small enough for a free T4 GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Keep Qwen's own pad token. If pad == eos, the collator masks the end token
# and the model never learns where an answer should stop.
assert tokenizer.pad_token != tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
)

## Baseline: the base model on the held-out questions

These answers come from the model **before** any fine-tuning.

In [ ]:
import re, string
from collections import Counter

def answer(question_context, question, max_new_tokens=40):
    inputs = tokenizer(make_prompt(question_context, question), return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip().split("\n")[0].strip()

def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))

def f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(p), common / len(g)
    return 2 * precision * recall / (precision + recall)

base_answers = [answer(c, q) for c, q, a in test_pairs]
for (c, q, a), pred in zip(test_pairs, base_answers):
    print(f"Q: {q}\n  base: {pred}\n  expected: {a}\n")

## Attach a LoRA adapter

LoRA trains a small set of adapter weights on the attention layers instead of the whole model.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Train

29 training pairs × 5 epochs, batch size 8 (2 × 4 accumulation) ≈ 20 optimizer steps. Loss is logged at every step so the curve is visible.

In [ ]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

train_dataset = Dataset.from_list([
    {"text": make_prompt(c, q) + a + tokenizer.eos_token} for c, q, a in train_pairs
])

def tokenize_fn(example):
    return tokenizer(example["text"], truncation=True, max_length=512)

tokenized_dataset = train_dataset.map(tokenize_fn, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="hospital-policy-qa-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator,
)
trainer.train()

## Before vs. after on the held-out topic

Same 5 questions, same passage the model never trained on. Scores: exact match (EM) and token F1, the standard reading-comprehension QA metrics.

In [ ]:
import pandas as pd

model.eval()
model.config.use_cache = True
tuned_answers = [answer(c, q) for c, q, a in test_pairs]

rows = []
for (c, q, a), b, t in zip(test_pairs, base_answers, tuned_answers):
    rows.append({
        "Question": q, "Expected": a,
        "Base model": b, "Base F1": round(f1(b, a), 2),
        "Fine-tuned": t, "Fine-tuned F1": round(f1(t, a), 2),
    })
results = pd.DataFrame(rows)

summary = pd.DataFrame({
    "Exact match": [sum(exact_match(b, a) for b, (_, _, a) in zip(base_answers, test_pairs)) / len(test_pairs),
                    sum(exact_match(t, a) for t, (_, _, a) in zip(tuned_answers, test_pairs)) / len(test_pairs)],
    "Token F1": [results["Base F1"].mean(), results["Fine-tuned F1"].mean()],
}, index=["Base model", "Fine-tuned"]).round(2)

pd.set_option("display.max_colwidth", None)
display(summary)
display(results)

## Save the adapter

In [ ]:
model.save_pretrained("hospital-policy-qa-lora")
tokenizer.save_pretrained("hospital-policy-qa-lora")

# Optional: publish the adapter to the Hugging Face Hub (needs a write token).
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("thumbup/qwen-hospital-policy-qa-lora")

## Limits

- 34 examples and a 0.5B model: a proof of concept, not a production assistant.
- The test set is 5 questions from one held-out topic, so scores move a lot with a single answer.
- Not for clinical use. A real hospital version would need the hospital's own policies, a much larger evaluation set, and staff review of every answer against the source document.